# Ablation Study - Fitur Mana yang Benar-Benar Menambah Akurasi

Baseline model (`02_baseline_model.ipynb`) memakai 16 fitur inti dan
mendapat PR-AUC 7,73% (CatBoost) di data validasi 2025 - sekitar 13x lebih
baik dari tebakan polos. Notebook ini menguji apakah menambah fitur
tambahan (lokasi, hierarki TERMINAL) benar-benar menaikkan akurasi secara
berarti, atau cuma menambah kerumitan tanpa manfaat nyata.

Caranya: fitur ditambahkan bertahap dalam 5 kelompok, bukan langsung semua
sekaligus, supaya jelas fitur kelompok mana yang menyumbang kenaikan dan
mana yang tidak. Model dilatih ulang dari nol untuk setiap kelompok memakai
pengaturan yang sama persis, supaya perbandingannya adil.


In [14]:
%pip install seaborn
%pip install sklearn


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached sklearn-0.0.post12.tar.gz (2.6 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [15 lines of output]
      The 'sklearn' PyPI package is deprecated, use 'scikit-learn'
      rather than 'sklearn' for pip commands.
      
      Here is how to fix this error in the main use cases:
      - use 'pip install scikit-learn' rather than 'pip install sklearn'
      - replace 'sklearn' by 'scikit-learn' in your pip requirements files
        (requirements.txt, setup.py, setup.cfg, Pipfile, etc ...)
      - if the 'sklearn' package is used by one of your dependencies,
        it would be great if you take some time to track which package uses
        'sklearn' instead of 'scikit-learn' and report it to their issue tracker
      - as a last resort, set the environment variable
        SKLEARN_ALLOW_DEPRECATED_SKLEARN_PACKAGE_INSTALL=True to avoid this error
      
      More information is available at
      https://github.com/scikit-learn/sklearn-pypi-packag

In [15]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.metrics import average_precision_score, roc_auc_score
from catboost import CatBoostClassifier, Pool

PROJECT_DIR = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_DIR / 'src'))
from database import connect
sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


def query(sql, params=None):
    with connect() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params or ())
            return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

ModuleNotFoundError: No module named 'sklearn'

## 0. Tujuan dan kelompok fitur yang diuji

**Tujuan:** menentukan kombinasi fitur paling sederhana yang tetap punya
sinyal prediktif kuat - bukan mengejar jumlah fitur sebanyak mungkin.

| Kelompok | Isi | Fitur baru yang ditambahkan |
|---|---|---|
| **A** | Model PART + umur | part_model_category, umur installation, bulan (siklik) |
| **B** | A + riwayat kerusakan/perbaikan | jumlah kejadian, jumlah kerusakan/corrective sebelumnya, hari sejak corrective terakhir |
| **C** | B + aktivitas terkini + klien (**= baseline penuh**, sama seperti `02_baseline_model.ipynb`) | jumlah lokasi berbeda, aktivitas 180 hari, klien |
| **D** | C + lokasi | lokasi terakhir, hari sejak kejadian/lokasi terakhir, interaksi model-lokasi, hari-dalam-minggu |
| **E** | D + hierarki TERMINAL (**= seluruh fitur challenger**) | jenis/model TERMINAL, status terakhir, interaksi model-TERMINAL |

Model E memakai seluruh 26 kolom di `analytics.failure_30d_challenger_features`.
Model dievaluasi di data validasi 2025 memakai PR-AUC dan Precision/Recall@K,
sama seperti baseline sebelumnya - bukan accuracy, karena target sangat
timpang.


In [ ]:
dataset = query("""
    SELECT f.*, l.target_failure_30d, l.temporal_split
    FROM analytics.failure_30d_challenger_features f
    JOIN analytics.failure_30d_model_labels l
      USING (installation_cycle_id, item_identifier_clean, observation_on)
    WHERE l.temporal_split IN ('TRAIN_2014_2024', 'VALIDATION_2025', 'TEST_2026')
""")

FEATURE_GROUPS = {
    'A - Model + umur': [
        'part_model_category', 'log_days_since_installation', 'installation_age_band',
        'month_sin', 'month_cos',
    ],
    'B - A + riwayat kerusakan': [
        'log_total_prior_events', 'log_prior_failure_count', 'has_prior_failure',
        'log_prior_corrective_count', 'has_prior_corrective', 'log_days_since_last_corrective',
        'log_prior_failure_365d', 'log_prior_corrective_30d',
    ],
    'C - B + aktivitas & klien (baseline penuh)': [
        'log_prior_distinct_places', 'log_prior_events_180d', 'client_category',
    ],
    'D - C + lokasi': [
        'location_category', 'log_days_since_last_event', 'log_days_at_last_location',
        'part_location_interaction', 'day_of_week_sin', 'day_of_week_cos',
    ],
    'E - D + hierarki TERMINAL (challenger penuh)': [
        'terminal_type_category', 'terminal_model_category', 'last_status_category',
        'part_terminal_type_interaction',
    ],
}
CATEGORICAL_COLUMNS = {
    'part_model_category', 'installation_age_band', 'client_category', 'location_category',
    'part_location_interaction', 'terminal_type_category', 'terminal_model_category',
    'last_status_category', 'part_terminal_type_interaction',
}

cumulative_columns = []
group_columns = {}
for group_name, new_columns in FEATURE_GROUPS.items():
    cumulative_columns = cumulative_columns + new_columns
    group_columns[group_name] = list(cumulative_columns)

all_columns = cumulative_columns
categorical_in_use = [c for c in all_columns if c in CATEGORICAL_COLUMNS]
numeric_in_use = [c for c in all_columns if c not in CATEGORICAL_COLUMNS]
dataset[categorical_in_use] = dataset[categorical_in_use].astype(str)
dataset[numeric_in_use] = dataset[numeric_in_use].apply(pd.to_numeric)
dataset['target_failure_30d'] = dataset['target_failure_30d'].astype(bool)

splits = {}
for split_name in ['TRAIN_2014_2024', 'VALIDATION_2025', 'TEST_2026']:
    part = dataset.loc[dataset.temporal_split.eq(split_name)]
    splits[split_name] = (part[all_columns], part['target_failure_30d'])
X_train_full, y_train = splits['TRAIN_2014_2024']
X_val_full, y_val = splits['VALIDATION_2025']
X_test_full, y_test = splits['TEST_2026']
display(Markdown(f"Data dimuat: **{len(y_train):,} baris train**, **{len(y_val):,} baris validasi**, **{len(y_test):,} baris test** (sama seperti baseline).".replace(',', '.')))

## 1. Latih CatBoost per kelompok fitur

Pengaturan model dibuat sama persis dengan `02_baseline_model.ipynb`
(`auto_class_weights='Balanced'`, early stopping di validasi) supaya
perbandingan antar kelompok adil - satu-satunya yang berubah hanyalah
fitur yang dipakai.


In [ ]:
def topk_table(y_true, y_proba, k_values):
    order = np.argsort(-y_proba)
    y_sorted = np.asarray(y_true)[order]
    total_positive = int(np.sum(y_true))
    rows = []
    for k in k_values:
        k = min(k, len(y_sorted))
        caught = int(y_sorted[:k].sum())
        rows.append({
            'K': k, 'Kerusakan tertangkap': caught,
            'Precision@K (%)': round(100.0 * caught / k, 2),
            'Recall@K (%)': round(100.0 * caught / total_positive, 2) if total_positive else 0.0,
        })
    return pd.DataFrame(rows)


def train_and_evaluate(group_name, columns):
    cats = [c for c in columns if c in CATEGORICAL_COLUMNS]
    train_pool = Pool(X_train_full[columns], y_train, cat_features=cats)
    val_pool = Pool(X_val_full[columns], y_val, cat_features=cats)
    model = CatBoostClassifier(
        iterations=1000, depth=6, learning_rate=0.05, loss_function='Logloss',
        eval_metric='PRAUC', auto_class_weights='Balanced', random_seed=RANDOM_STATE,
        early_stopping_rounds=50, verbose=False,
    )
    model.fit(train_pool, eval_set=val_pool)
    val_proba = model.predict_proba(X_val_full[columns])[:, 1]
    return model, val_proba


results = {}
models_by_group = {}
for group_name, columns in group_columns.items():
    model, val_proba = train_and_evaluate(group_name, columns)
    models_by_group[group_name] = model
    results[group_name] = {
        'Jumlah fitur mentah': len(columns),
        'PR-AUC': average_precision_score(y_val, val_proba),
        'ROC-AUC': roc_auc_score(y_val, val_proba),
        'val_proba': val_proba,
        'best_iteration': model.get_best_iteration(),
    }
    print(f"[selesai] {group_name}: PR-AUC={results[group_name]['PR-AUC']:.4%}, iterasi terbaik={model.get_best_iteration()}")

## 2. Perbandingan bertahap: fitur mana yang benar-benar menaikkan akurasi?

Kenaikan PR-AUC dari satu kelompok ke kelompok berikutnya menunjukkan
apakah fitur yang baru ditambahkan itu benar-benar berguna. Kalau
kenaikannya kecil atau malah turun, berarti fitur tersebut tidak sepadan
dengan tambahan kerumitannya.


In [ ]:
comparison = pd.DataFrame([
    {'Kelompok': name, 'Jumlah fitur mentah': r['Jumlah fitur mentah'],
     'PR-AUC': r['PR-AUC'], 'ROC-AUC': r['ROC-AUC']}
    for name, r in results.items()
])
comparison['Kenaikan PR-AUC vs sebelumnya (poin persen)'] = (
    comparison['PR-AUC'].diff().fillna(comparison['PR-AUC'].iloc[0]) * 100
).round(3)
display(comparison.style.format({'PR-AUC': '{:.4%}', 'ROC-AUC': '{:.4f}'}))

plt.figure(figsize=(8, 5))
sns.lineplot(data=comparison, x='Kelompok', y='PR-AUC', marker='o', sort=False)
plt.xticks(rotation=25, ha='right')
plt.ylabel('PR-AUC di validasi 2025')
plt.title('PR-AUC bertambah seiring penambahan kelompok fitur')
plt.tight_layout(); plt.show()

for name, r in results.items():
    display(Markdown(f"**Precision/Recall@K - {name}**"))
    display(topk_table(y_val, r['val_proba'], [100, 500, 1000, 5000]))

## 3. Konfirmasi akhir di data test (2026)

Kelompok dengan PR-AUC validasi tertinggi dikonfirmasi sekali lagi di data
test 2026, dibandingkan dengan baseline penuh (kelompok C) sebagai
pembanding utama.


In [ ]:
best_group = comparison.sort_values('PR-AUC', ascending=False).iloc[0]['Kelompok']
baseline_group = 'C - B + aktivitas & klien (baseline penuh)'

test_rows = []
for group_name in [baseline_group, best_group] if best_group != baseline_group else [baseline_group]:
    columns = group_columns[group_name]
    model = models_by_group[group_name]
    test_proba = model.predict_proba(X_test_full[columns])[:, 1]
    test_rows.append({
        'Kelompok': group_name,
        'PR-AUC (test 2026)': average_precision_score(y_test, test_proba),
        'ROC-AUC (test 2026)': roc_auc_score(y_test, test_proba),
    })
display(pd.DataFrame(test_rows).style.format({'PR-AUC (test 2026)': '{:.4%}', 'ROC-AUC (test 2026)': '{:.4f}'}))

## 4. Kesimpulan dan rekomendasi fitur


In [ ]:
baseline_praucs = results[baseline_group]['PR-AUC']
best_prauc = comparison.sort_values('PR-AUC', ascending=False).iloc[0]['PR-AUC']
best_group_row = comparison.sort_values('PR-AUC', ascending=False).iloc[0]
lift_vs_baseline = (best_prauc / baseline_praucs - 1) * 100 if baseline_praucs else float('nan')
worthwhile = comparison.loc[comparison['Kenaikan PR-AUC vs sebelumnya (poin persen)'] > 0.3, 'Kelompok'].tolist()
not_worthwhile = comparison.loc[comparison['Kenaikan PR-AUC vs sebelumnya (poin persen)'] <= 0.3, 'Kelompok'].tolist()

display(Markdown(f"""**Ringkasan ablation study**

- PR-AUC baseline penuh (kelompok C, 16 fitur): **{baseline_praucs:.4%}**.
- PR-AUC tertinggi dicapai oleh **{best_group_row['Kelompok']}**: **{best_prauc:.4%}**.
- Selisihnya: **{lift_vs_baseline:+.1f}%** dibanding baseline penuh.
- Kelompok dengan kenaikan PR-AUC yang cukup berarti (>0,3 poin persen dari kelompok sebelumnya): {', '.join(worthwhile) if worthwhile else 'tidak ada'}.
- Kelompok dengan kenaikan kecil/tidak berarti: {', '.join(not_worthwhile) if not_worthwhile else 'tidak ada'}.

**Cara membaca hasil ini:** kalau kelompok D (lokasi) atau E (hierarki TERMINAL) ternyata menaikkan PR-AUC secara berarti, berarti fitur tambahan itu layak dipakai produksi - meskipun tetap harus lolos sensitivity analysis dulu (terutama fitur TERMINAL yang sebagian relasinya direkonstruksi belakangan). Kalau kenaikannya kecil, lebih baik tetap pakai baseline (kelompok C) yang lebih sederhana dan lebih mudah dijelaskan/dirawat, sesuai prinsip "fitur paling sederhana yang tetap kuat" - bukan "fitur sebanyak mungkin".

**Langkah selanjutnya:** kelompok pemenang di atas masih perlu diuji lewat sensitivity analysis (`is_strict_training_eligible`) sebelum dipertimbangkan untuk produksi."""))